# Autoencoders & Latent Spaces

This notebook accompanies the **ML Viz** lesson on autoencoders.
We'll build a vanilla autoencoder from scratch and explore the latent space.

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/generative-models/02-autoencoders

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## Intuition — compress, then reconstruct

An **autoencoder** learns a compressed representation by being forced through a bottleneck: an
**encoder** maps data `x` to a low-dimensional latent `z`, a **decoder** maps `z` back to `x̂`, and
training minimizes the **reconstruction error** `‖x − x̂‖²`. To reconstruct well through a narrow `z`,
the network must discover the data's underlying structure — its manifold. That makes autoencoders great
for representation learning and compression. But as a *generative* model they have a fatal gap: nothing
organizes the latent space, so decoding a random `z` lands in a **hole** and produces garbage — the
precise problem VAEs (next lesson) fix. We train one from scratch on a swiss roll and validate against
the theoretical optimum.

## Autoencoder architecture

An autoencoder has two parts:
- **Encoder**: compresses input $x$ into a latent vector $z$
- **Decoder**: reconstructs $x$ from $z$

We'll implement this using only NumPy — no frameworks needed.

In [ ]:
class Autoencoder:
    """Simple linear autoencoder with ReLU activations."""
    
    def __init__(self, input_dim, latent_dim):
        scale = np.sqrt(2.0 / input_dim)
        self.W_enc = np.random.randn(input_dim, latent_dim) * scale
        self.b_enc = np.zeros(latent_dim)
        self.W_dec = np.random.randn(latent_dim, input_dim) * scale
        self.b_dec = np.zeros(input_dim)
    
    def relu(self, x):
        return np.maximum(0, x)
    
    def encode(self, x):
        self.z_pre = x @ self.W_enc + self.b_enc
        self.z = self.relu(self.z_pre)
        return self.z
    
    def decode(self, z):
        self.x_hat = z @ self.W_dec + self.b_dec
        return self.x_hat
    
    def forward(self, x):
        z = self.encode(x)
        return self.decode(z)
    
    def loss(self, x):
        x_hat = self.forward(x)
        return np.mean((x - x_hat) ** 2)
    
    def backward(self, x, lr=0.01):
        n = x.shape[0]
        x_hat = self.forward(x)
        
        # Gradient of MSE loss w.r.t. output
        dx_hat = 2 * (x_hat - x) / n
        
        # Decoder gradients
        dW_dec = self.z.T @ dx_hat
        db_dec = dx_hat.sum(axis=0)
        
        # Through ReLU
        dz = dx_hat @ self.W_dec.T
        dz_pre = dz * (self.z_pre > 0).astype(float)  # ReLU gradient
        
        # Encoder gradients
        dW_enc = x.T @ dz_pre
        db_enc = dz_pre.sum(axis=0)
        
        # Update
        self.W_enc -= lr * dW_enc
        self.b_enc -= lr * db_enc
        self.W_dec -= lr * dW_dec
        self.b_dec -= lr * db_dec

print('Autoencoder class defined.')

**What to notice:** the architecture is two small networks glued at the bottleneck — `encode` (3D→2D
with ReLU) and `decode` (2D→3D, linear) — trained end-to-end by backprop on MSE. The gradient code is
the same chain rule as the MLP notebooks; the only novelty is that the *target is the input itself*.

## Generate 2D Swiss Roll data

A Swiss roll is a classic dataset where a 2D manifold is embedded in higher-dimensional space.
The autoencoder should learn to unroll it.

In [ ]:
np.random.seed(42)
n_points = 500

t = 1.5 * np.pi * (1 + 2 * np.random.rand(n_points))
x_swiss = np.column_stack([
    t * np.cos(t),
    t * np.sin(t),
    30 * np.random.rand(n_points)  # noise in 3rd dimension
])

# Normalize
x_swiss = (x_swiss - x_swiss.mean(axis=0)) / x_swiss.std(axis=0)

fig = plt.figure(figsize=(8, 5))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(x_swiss[:, 0], x_swiss[:, 1], x_swiss[:, 2], c=t, cmap='viridis', s=10, alpha=0.7)
ax.set_title('Swiss Roll Dataset', color='white', fontsize=12)
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_zlabel('$x_3$')
ax.view_init(elev=20, azim=45)
plt.tight_layout()
plt.show()

**What to notice:** the swiss roll is 3-D data lying on an intrinsically **2-D** surface — exactly
the situation autoencoders exploit. A 2-D bottleneck is information-theoretically enough *if* the
network can unroll the manifold.

## Train the autoencoder

We'll compress from 3D to 2D (the true intrinsic dimensionality of the Swiss roll).

In [ ]:
ae = Autoencoder(input_dim=3, latent_dim=2)

losses = []
for epoch in range(500):
    loss = ae.loss(x_swiss)
    losses.append(loss)
    ae.backward(x_swiss, lr=0.005)
    if (epoch + 1) % 100 == 0:
        print(f'Epoch {epoch+1:3d} | MSE Loss: {loss:.4f}')

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(losses, color='#818cf8', linewidth=1.5)
ax.set_title('Training Loss', color='white', fontsize=12)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
plt.tight_layout()
plt.show()

**What to notice:** the MSE falls smoothly — the bottleneck is learning to preserve what matters.
The loss can't reach zero: a 2-D linear-ish latent can't perfectly capture the curled 3-D roll, so the
floor reflects the mismatch between model capacity and manifold curvature.

## The library way — validate against the PCA optimum

For a **linear** decoder with MSE loss, the best possible rank-2 reconstruction is exactly **PCA's**
(Eckart–Young, from the SVD lesson). So PCA's rank-2 error is a hard floor for our autoencoder — the
check: our trained AE's error must be ≥ the PCA floor, and reasonably close to it.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2).fit(x_swiss)
x_pca = pca.inverse_transform(pca.transform(x_swiss))
pca_mse = np.mean((x_swiss - x_pca) ** 2)
ae_mse = ae.loss(x_swiss)

print(f'PCA rank-2 reconstruction MSE (theoretical floor): {pca_mse:.4f}')
print(f'our autoencoder reconstruction MSE               : {ae_mse:.4f}')
assert ae_mse >= pca_mse - 1e-9, "no linear-decoder AE can beat the PCA floor (Eckart-Young)"
assert ae_mse < 3 * pca_mse + 0.1, "a trained AE should be in the same ballpark as PCA"
print('\nAE error >= PCA floor, and close to it: the bottleneck learned the principal subspace ✓')

**What to notice:** the autoencoder's error sits just above PCA's rank-2 floor — as Eckart–Young
demands for a linear decoder. A linear autoencoder *is* PCA (up to rotation of the latent axes); only
**deep, non-linear** encoders/decoders can beat this floor by bending the subspace along the manifold.
That's the entire motivation for making autoencoders deep.

## Visualize the latent space

The encoder should map the 3D Swiss roll into a clean 2D representation.

In [ ]:
z_encoded = ae.encode(x_swiss)
x_reconstructed = ae.forward(x_swiss)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Original 3D
ax = fig.add_subplot(131, projection='3d')
ax.scatter(x_swiss[:, 0], x_swiss[:, 1], x_swiss[:, 2], c=t, cmap='viridis', s=8, alpha=0.6)
ax.set_title('Original (3D)', color='white', fontsize=11)
ax.view_init(elev=20, azim=45)

# Latent space
axes[1].scatter(z_encoded[:, 0], z_encoded[:, 1], c=t, cmap='viridis', s=10, alpha=0.7)
axes[1].set_title('Latent Space (2D)', color='white', fontsize=11)
axes[1].set_xlabel('$z_1$')
axes[1].set_ylabel('$z_2$')

# Reconstruction
ax = fig.add_subplot(133, projection='3d')
ax.scatter(x_reconstructed[:, 0], x_reconstructed[:, 1], x_reconstructed[:, 2], c=t, cmap='viridis', s=8, alpha=0.6)
ax.set_title('Reconstruction', color='white', fontsize=11)
ax.view_init(elev=20, azim=45)

plt.tight_layout()
plt.show()

**What to notice:** the 2-D latent space lays the swiss roll out — nearby latent points decode to
nearby data points, and the roll's parameterization (color) varies smoothly across the latent plane.
The bottleneck has *discovered coordinates on the manifold* without ever being told they exist.

## The problem: latent space holes

A standard autoencoder maps training inputs to points, but the latent space
between them is unstructured. Sampling randomly produces garbage.

In [ ]:
# Sample random points from latent space
np.random.seed(7)
z_random = np.random.randn(16, 2) * 1.5  # wider than the encoded distribution
x_random = z_random @ ae.W_dec + ae.b_dec  # decode directly

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
fig.suptitle('Random Latent Samples → Decoded (Unstructured = Garbage)', color='white', fontsize=12, y=1.02)

for i in range(16):
    ax = axes[i // 8, i % 8]
    # Show as a simple bar chart of the 3D output
    ax.bar(range(3), x_random[i], color='#f43f5e', alpha=0.8)
    ax.set_ylim(-3, 3)
    ax.axis('off')

plt.tight_layout()
plt.show()

print('These random samples don\'t look like the original data.')
print('The latent space has holes — the decoder was never trained on these regions.')
print('This is why we need VAEs (next lesson) to structure the latent space.')

**What to notice:** decoding **random** latent points produces outputs that look nothing like the
data — the encoder only populated part of the latent plane, and the decoder is undefined off that
support. This is the **latent holes** problem: an autoencoder is a great compressor but a broken
*generator*, because nothing forces the latent space to be a nice, samplable distribution. The VAE's
KL term (next lesson) exists precisely to fill these holes.

## Gotchas & tradeoffs

- **Autoencoders ≠ generators.** Without a latent prior, random `z` decodes to garbage — use VAEs (or
  train a prior over the latents) for generation.
- **Too-wide bottlenecks learn the identity.** If `latent_dim ≥ input_dim`, the network can copy
  without compressing; the bottleneck (or noise/sparsity penalties) is the whole regularizer.
- **A linear AE can't beat PCA** — depth/non-linearity is what buys anything beyond the principal
  subspace.
- **Reconstruction MSE favors blur.** Pixel-space MSE penalizes misplaced sharpness more than smooth
  averages — why plain-AE images look soft (and why perceptual/adversarial losses exist).

In [ ]:
# Too-wide bottleneck: latent_dim >= input_dim lets the AE approach the identity map (no compression)
np.random.seed(0)
ae_wide = Autoencoder(input_dim=3, latent_dim=3)
for _ in range(5000):
    ae_wide.backward(x_swiss, lr=0.01)
print(f'latent_dim=2 (bottleneck) MSE: {ae.loss(x_swiss):.4f}')
print(f'latent_dim=3 (no bottleneck) MSE: {ae_wide.loss(x_swiss):.4f}  -> approaching the identity map')
print('-> without a bottleneck the AE learns to copy; the compression IS the regularizer')

**What to notice:** with `latent_dim = input_dim` the reconstruction error collapses toward zero —
the network is just learning to copy its input, extracting no structure. The information bottleneck is
what forces representation learning; remove it and an autoencoder learns nothing worth keeping.

## Key takeaways

1. Autoencoders learn to compress and reconstruct data
2. The bottleneck forces the network to learn meaningful representations
3. But the latent space is unstructured — random sampling produces garbage
4. **Next lesson:** VAEs fix this by forcing the latent space to follow $\mathcal{N}(0, I)$

## ✏️ Your turn

### Exercise 1 — MSE reconstruction loss

The autoencoder's training objective is the mean squared reconstruction error:
$$\mathcal{L} = \frac{1}{n} \sum_{i=1}^n \| x_i - \hat{x}_i \|^2$$

Implement it and verify: non-negative, zero for perfect reconstruction, and symmetric.

In [ ]:
import numpy as np

def mse_loss(x, x_hat):
    """Mean squared error between x and x_hat (both shape (N, d))."""
    # TODO(you): implement MSE
    ...

In [ ]:
x     = np.array([[1.0, 2.0], [3.0, 4.0]])
x_hat = np.array([[1.5, 1.5], [3.0, 4.5]])

loss = mse_loss(x, x_hat)

assert loss >= 0, "MSE must be non-negative"
assert mse_loss(x, x) == 0.0, \
    "MSE is exactly 0 when reconstruction equals input"
assert abs(mse_loss(x, x_hat) - mse_loss(x_hat, x)) < 1e-12, \
    "MSE is symmetric: MSE(x, x_hat) == MSE(x_hat, x)"
# manual check: differences are [0.5,-0.5] and [0.0,0.5]; squared=[0.25,0.25,0.0,0.25]; mean=0.1875
assert abs(loss - 0.1875) < 1e-9, "expected MSE = 0.1875"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def mse_loss(x, x_hat):
    return np.mean((x - x_hat) ** 2)
```

</details>

### Exercise 2 — ReLU forward and gradient mask

ReLU is elementwise: $\text{ReLU}(z) = \max(0, z)$. Its sub-gradient is 1 where $z>0$ and 0
elsewhere. Implement both and verify the sign-based properties.

In [ ]:
def relu_forward(z):
    """Elementwise ReLU: max(0, z). Returns same shape as z."""
    # TODO(you): one line
    ...

def relu_grad_mask(z):
    """Sub-gradient of ReLU: 1 where z > 0, 0 elsewhere (including z = 0)."""
    # TODO(you): one line
    ...

In [ ]:
z = np.array([-3.0, -1.0, 0.0, 2.0, 4.0])

out  = relu_forward(z)
grad = relu_grad_mask(z)

assert np.all(out >= 0), "ReLU output must be non-negative"
assert out[0] == 0 and out[1] == 0 and out[2] == 0, \
    "ReLU must zero-out negative and zero inputs"
assert out[3] == 2.0 and out[4] == 4.0, \
    "ReLU passes positive values unchanged"
assert grad[2] == 0, \
    "sub-gradient at z=0 is 0 (convention used in backprop)"
assert np.array_equal(grad.astype(bool), out.astype(bool)), \
    "gradient mask is 1 exactly where output is positive"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def relu_forward(z):
    return np.maximum(0, z)

def relu_grad_mask(z):
    return (z > 0).astype(float)
```

</details>